# SNI-21 Density Evaluation — frozen A0 on R0/B0/B1/B2/B3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ediprin/coffee-bean-detection/blob/agent/add-vadcp-pipeline/notebooks/SNI21_Density_Evaluation_Colab.ipynb)

Pilih runtime **T4 GPU**, lalu jalankan semua sel.

Notebook ini:

- memakai checkpoint A0 seed 42 yang sudah dibekukan;
- mengevaluasi synthetic development B0–B3 dan validation nyata R0 jika archive A0 tersedia;
- memakai konfigurasi inference identik pada semua kondisi;
- memisahkan proposal miss, localized wrong-class, dan saturasi `max_det`;
- tidak menjalankan training;
- tidak mengekstrak atau membaca gambar test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
REPOSITORY = 'https://github.com/ediprin/coffee-bean-detection.git'

if not (REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        REPOSITORY, str(REPO),
    ], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)

import torch
assert torch.cuda.is_available(), 'Aktifkan runtime T4 GPU sebelum evaluasi.'
print('GPU   :', torch.cuda.get_device_name(0))
print('REPO  :', REPO)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO, check=True)

In [ ]:
DRIVE = Path('/content/drive/MyDrive')
SEARCH_ROOTS = [
    DRIVE,
    Path('/content/drive/.shortcut-targets-by-id'),
    Path('/content/drive/Shareddrives'),
]

def find_artifact(preferred, pattern, label, required=True):
    if preferred.is_file():
        return preferred
    suffix = Path(pattern).parts
    matches = []
    for search_root in SEARCH_ROOTS:
        if not search_root.is_dir():
            continue
        for path in search_root.rglob(Path(pattern).name):
            if path.is_file() and tuple(path.parts[-len(suffix):]) == suffix:
                matches.append(path)
    matches = sorted(set(matches))
    if not matches:
        message = (
            f'{label} tidak ditemukan di MyDrive. Pastikan folder yang dibagikan ' 
            'sudah ditambahkan sebagai shortcut ke My Drive.'
        )
        if required:
            raise FileNotFoundError(message)
        print('OPSIONAL -', message)
        return None
    if len(matches) > 1:
        print(f'{label}: ditemukan {len(matches)} kandidat; memakai {matches[0]}')
    return matches[0]

CHECKPOINT = find_artifact(
    DRIVE / 'coffee-bean-detection/sni21-vadcp-pilot-results/A0_seed42/weights/best.pt',
    'A0_seed42/weights/best.pt',
    'Checkpoint A0',
)
A0_ARCHIVE = find_artifact(
    DRIVE / 'coffee-bean-detection/sni21-vadcp-pilot-bundle/A0_real.tar',
    'A0_real.tar',
    'Archive A0',
    required=False,
)
BENCHMARK_SUMMARY = find_artifact(
    DRIVE / 'coffee-bean-detection/sni21-density-benchmark-v1/setup_core_summary.json',
    'sni21-density-benchmark-v1/setup_core_summary.json',
    'Summary benchmark',
)
BENCHMARK_ROOT = BENCHMARK_SUMMARY.parent
OUTPUT_ROOT = BENCHMARK_ROOT.parent / 'sni21-density-evaluation-v1'
R0_ROOT = Path('/content/sni21-fullscene-v1')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('CHECKPOINT:', CHECKPOINT)
print('A0 ARCHIVE:', A0_ARCHIVE if A0_ARCHIVE else 'TIDAK ADA - R0 dilewati sementara')
print('BENCHMARK :', BENCHMARK_ROOT)
print('OUTPUT    :', OUTPUT_ROOT)

In [ ]:
from coffee_detector.archive_sni21_pilot import restore_real_a0_validation

REAL_ROOT_ARGS = []
if A0_ARCHIVE is None:
    print('R0 DILEWATI: evaluasi tetap berjalan pada B0-B3.')
    print('Archive dapat dibagikan belakangan; hasil B0-B3 akan dipakai ulang.')
else:
    restore_real_a0_validation(A0_ARCHIVE, R0_ROOT)
    restore_report = json.loads((R0_ROOT / 'validation_restore.json').read_text(encoding='utf-8'))
    assert restore_report['test_files_extracted'] == 0
    assert restore_report['test_images_accessed'] is False
    REAL_ROOT_ARGS = ['--real-root', str(R0_ROOT)]
    print(json.dumps(restore_report, indent=2, ensure_ascii=False))

In [ ]:
command = [
    sys.executable, '-u', '-m',
    'coffee_detector.run_sni21_density_evaluation',
    '--checkpoint', str(CHECKPOINT),
    *REAL_ROOT_ARGS,
    '--benchmark-root', str(BENCHMARK_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--device', '0',
    '--imgsz', '640',
    '--batch-size', '8',
    '--confidence', '0.001',
    '--nms-iou', '0.7',
    '--diagnostic-iou', '0.5',
    '--max-det', '300',
]

print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    cwd=REPO,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
assert return_code == 0, f'Evaluasi gagal dengan return code {return_code}'
print('\nEVALUASI SELESAI')

In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY_PATH = OUTPUT_ROOT / 'density_evaluation_summary.json'
assert SUMMARY_PATH.is_file(), f'Summary belum ditemukan: {SUMMARY_PATH}'
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
assert summary['training_executed'] is False
assert summary['test_images_accessed'] is False

table = pd.DataFrame(summary['rows'])
percentage_columns = [
    'map50_95', 'map50', 'precision', 'recall',
    'macro_map50_95', 'bottom3_map50_95', 'worst_map50_95',
    'proposal_recall_at_50', 'conditional_class_accuracy',
    'proposal_miss_rate', 'localized_wrong_class_rate',
    'saturation_rate',
]
styled = table.style.format({column: '{:.2%}' for column in percentage_columns})
display(styled)
print('SUMMARY:', SUMMARY_PATH)
print('TRAINING:', summary['training_executed'])
print('TEST ACCESSED:', summary['test_images_accessed'])
print('\nKirim tabel ini sebelum mencoba max_det lain atau melatih model baru.')